# Transformación de Bronce a Plata

In [31]:
spark.sparkContext.setLogLevel("ERROR")

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("ETL-Bronce-Plata") \
    .getOrCreate()

spark

25/12/10 03:26:11 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


## Cargando los datos del csv

### Diabetes

In [2]:
df_bronce_diabetes = spark.read.csv(
    "gs://grupo2-essalud-datalake/bronce/Diabetes.csv",
    header=True,
    inferSchema=True,
    sep=";"
)

df_bronce_diabetes.show(5)
df_bronce_diabetes.printSchema()

25/12/10 03:26:26 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-----------+------------+----------------+-----------+------+--------------------+--------------------+--------------------+-------------+-------------+-----------+--------------------+--------+--------------------+-----------------+---------------------+----------------------+-------------+---------------+--------------------+-----------+----------+---------------+--------------------+-----------+----------+
|FECHA_CORTE|DEPARTAMENTO|       PROVINCIA|   DISTRITO|UBIGEO|                 RED|              IPRESS|         ID_PACIENTE|EDAD_PACIENTE|SEXO_PACIENTE|EDAD_MEDICO|           ID_MEDICO|COD_DIAG|         DIAGNOSTICO|AREA_HOSPITALARIA|SERVICIO_HOSPITALARIO|ACTIVIDAD_HOSPITALARIA|FECHA_MUESTRA|FEC_RESULTADO_1|     PROCEDIMIENTO_1|RESULTADO_1|UNIDADES_1|FEC_RESULTADO_2|     PROCEDIMIENTO_2|RESULTADO_2|UNIDADES_2|
+-----------+------------+----------------+-----------+------+--------------------+--------------------+--------------------+-------------+-------------+-----------+-------

### Hipertension

In [3]:
df_bronce_hipertension = spark.read.csv(
    "gs://grupo2-essalud-datalake/bronce/Hipertension.csv",
    header=True,
    inferSchema=True,
    sep=";"
)

df_bronce_hipertension.show(5)
df_bronce_hipertension.printSchema()

+-----------+------------+----------------+-----------+------+--------------------+-----------------+--------------------+-------------+-------------+-----------+--------------------+--------+--------------------+-----------------+---------------------+----------------------+-------------+---------------+--------------------+-----------+----------+---------------+---------------+-----------+----------+
|FECHA_CORTE|DEPARTAMENTO|       PROVINCIA|   DISTRITO|UBIGEO|                 RED|           IPRESS|         ID_PACIENTE|EDAD_PACIENTE|SEXO_PACIENTE|EDAD_MEDICO|           ID_MEDICO|COD_DIAG|         DIAGNOSTICO|AREA_HOSPITALARIA|SERVICIO_HOSPITALARIO|ACTIVIDAD_HOSPITALARIA|FECHA_MUESTRA|FEC_RESULTADO_1|     PROCEDIMIENTO_1|RESULTADO_1|UNIDADES_1|FEC_RESULTADO_2|PROCEDIMIENTO_2|RESULTADO_2|UNIDADES_2|
+-----------+------------+----------------+-----------+------+--------------------+-----------------+--------------------+-------------+-------------+-----------+--------------------+-----

### Obesidad

In [4]:
df_bronce_obesidad = spark.read.csv(
    "gs://grupo2-essalud-datalake/bronce/Obesidad.csv",
    header=True,
    inferSchema=True,
    sep=";"
)

df_bronce_obesidad.show(5)
df_bronce_obesidad.printSchema()

+-----------+------------+---------+----------+------+--------------------+--------------------+--------------------+-------------+-------------+-----------+--------------------+--------+--------------------+-----------------+---------------------+----------------------+-------------+---------------+--------------------+-----------+----------+---------------+--------------------+-----------+----------+
|FECHA_CORTE|DEPARTAMENTO|PROVINCIA|  DISTRITO|UBIGEO|                 RED|              IPRESS|         ID_PACIENTE|EDAD_PACIENTE|SEXO_PACIENTE|EDAD_MEDICO|           ID_MEDICO|COD_DIAG|         DIAGNOSTICO|AREA_HOSPITALARIA|SERVICIO_HOSPITALARIO|ACTIVIDAD_HOSPITALARIA|FECHA_MUESTRA|FEC_RESULTADO_1|     PROCEDIMIENTO_1|RESULTADO_1|UNIDADES_1|FEC_RESULTADO_2|     PROCEDIMIENTO_2|RESULTADO_2|UNIDADES_2|
+-----------+------------+---------+----------+------+--------------------+--------------------+--------------------+-------------+-------------+-----------+--------------------+--------+-

## Carga de datos Bronce a Big Query

In [5]:
print("Diabetes:", df_bronce_diabetes.count())
print("Hipertensión:", df_bronce_hipertension.count())
print("Obesidad:", df_bronce_obesidad.count())

Diabetes: 509716
Hipertensión: 503638
Obesidad: 294971


Hipertensión: 503638
Obesidad: 294971


In [8]:
df_bronce_diabetes.write.format("bigquery") \
    .option("table", "grupo2-essalud.bronce.diabetes") \
    .option("temporaryGcsBucket", "grupo2-essalud-datalake") \
    .mode("overwrite") \
    .save()

df_bronce_hipertension.write.format("bigquery") \
    .option("table", "grupo2-essalud.bronce.hipertension") \
    .option("temporaryGcsBucket", "grupo2-essalud-datalake") \
    .mode("overwrite") \
    .save()

df_bronce_obesidad.write.format("bigquery") \
    .option("table", "grupo2-essalud.bronce.obesidad") \
    .option("temporaryGcsBucket", "grupo2-essalud-datalake") \
    .mode("overwrite") \
    .save()

## Transformación de Datos de Bronce a Plata

In [29]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sparkContext.setLogLevel("ERROR")

### Procedimiento

In [15]:
df_union = (
    df_bronce_diabetes.select(F.col("PROCEDIMIENTO_1").alias("des_procedimiento"))
    .union(df_bronce_diabetes.select(F.col("PROCEDIMIENTO_2").alias("des_procedimiento")))
    .union(df_bronce_hipertension.select(F.col("PROCEDIMIENTO_1").alias("des_procedimiento")))
    .union(df_bronce_hipertension.select(F.col("PROCEDIMIENTO_2").alias("des_procedimiento")))
    .union(df_bronce_obesidad.select(F.col("PROCEDIMIENTO_1").alias("des_procedimiento")))
    .union(df_bronce_obesidad.select(F.col("PROCEDIMIENTO_2").alias("des_procedimiento")))
)

df_distinct = df_union.filter(F.col("des_procedimiento").isNotNull()).dropDuplicates()

df_plata_procedimiento = df_distinct.withColumn(
    "cod_procedimiento",
    F.row_number().over(Window.orderBy("des_procedimiento"))
)

df_plata_procedimiento.show(20, truncate=False)
df_plata_procedimiento.printSchema()

25/12/10 04:04:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 04:04:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 04:04:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+------------------------------------------------------------------+-----------------+
|des_procedimiento                                                 |cod_procedimiento|
+------------------------------------------------------------------+-----------------+
|DOSAJE DE COLESTEROL TOTAL EN SANGRE COMPLETA O SUERO             |1                |
|DOSAJE DE CREATININA EN SANGRE                                    |2                |
|DOSAJE DE GLUCOSA EN SANGRE, CUANTITATIVO (EXCEPTO CINTA REACTIVA)|3                |
|TRIGLICERIDOS                                                     |4                |
+------------------------------------------------------------------+-----------------+

root
 |-- des_procedimiento: string (nullable = true)
 |-- cod_procedimiento: integer (nullable = false)



### Medico

In [18]:
df_plata_medico = (
    df_bronce_diabetes.select(
        F.col("ID_MEDICO").alias("cod_medico"),
        F.col("EDAD_MEDICO").alias("edad_medico")
    )
    .union(
        df_bronce_hipertension.select(
            F.col("ID_MEDICO").alias("cod_medico"),
            F.col("EDAD_MEDICO").alias("edad_medico")
        )
    )
    .union(
        df_bronce_obesidad.select(
            F.col("ID_MEDICO").alias("cod_medico"),
            F.col("EDAD_MEDICO").alias("edad_medico")
        )
    )
    .dropDuplicates()
)

df_plata_medico.show(10)

[Stage 31:===================================================>    (22 + 2) / 24]

+--------------------+-----------+
|          cod_medico|edad_medico|
+--------------------+-----------+
|eJwzNDA0MzIxsLA0N...|         67|
|eJwzNLQwNbA0MbMwM...|         64|
|eJwzNDAyMzewNDAyM...|         46|
|eJwzNLMwMDU3NTYwM...|         46|
|eJwzNjA2NjIxMzM2N...|         25|
|eJwzNDQ1sTC2MDU2A...|         52|
|eJwzMjAwMzMwNjK3M...|         38|
|eJwzsjQxMjA1NzA3t...|         26|
|eJwzNDAyM7UwM7G0N...|         48|
|eJwzMjQ2MTQ0MDI0N...|         32|
+--------------------+-----------+
only showing top 10 rows



### Paciente

In [19]:
df_plata_paciente = (
    df_bronce_diabetes.select(
        F.col("ID_PACIENTE").alias("cod_paciente"),
        F.col("EDAD_PACIENTE").alias("edad_paciente"),
        F.col("SEXO_PACIENTE").alias("sexo_paciente")
    )
    .union(
        df_bronce_hipertension.select(
            F.col("ID_PACIENTE").alias("cod_paciente"),
            F.col("EDAD_PACIENTE").alias("edad_paciente"),
            F.col("SEXO_PACIENTE").alias("sexo_paciente")
        )
    )
    .union(
        df_bronce_obesidad.select(
            F.col("ID_PACIENTE").alias("cod_paciente"),
            F.col("EDAD_PACIENTE").alias("edad_paciente"),
            F.col("SEXO_PACIENTE").alias("sexo_paciente")
        )
    )
    .dropDuplicates()
)

df_plata_paciente.show(10)

[Stage 34:=====================================================>  (23 + 1) / 24]

+--------------------+-------------+-------------+
|        cod_paciente|edad_paciente|sexo_paciente|
+--------------------+-------------+-------------+
|eJwzNDQxNzIwN7QwM...|           58|     FEMENINO|
|eJwzNDKwMDA3MDMxN...|           45|    MASCULINO|
|eJwzNDc0NLEwMjEyt...|           60|    MASCULINO|
|eJwzNDCwMDWwNDEwt...|           72|     FEMENINO|
|eJwzNDQ2M7A0tbCwN...|           73|     FEMENINO|
|eJwzNLUwMLK0NDa3N...|           50|    MASCULINO|
|eJwzNjKxNDU2NjA3M...|           56|     FEMENINO|
|eJwzNDQzNjYyNTQ1M...|           60|     FEMENINO|
|eJwzNLQwMTYwNDC3M...|           68|     FEMENINO|
|eJwzNDQyM7EwMzezN...|           60|    MASCULINO|
+--------------------+-------------+-------------+
only showing top 10 rows



### Enfermedad

In [20]:
df_plata_enfermedad = (
    df_bronce_diabetes.select(
        F.col("COD_DIAG").alias("cod_enfermedad"),
        F.col("DIAGNOSTICO").alias("enfermedad"),
        F.lit("Diabetes").alias("grupo_enfermedad")
    )
    .union(
        df_bronce_hipertension.select(
            F.col("COD_DIAG").alias("cod_enfermedad"),
            F.col("DIAGNOSTICO").alias("enfermedad"),
            F.lit("Hipertension").alias("grupo_enfermedad")
        )
    )
    .union(
        df_bronce_obesidad.select(
            F.col("COD_DIAG").alias("cod_enfermedad"),
            F.col("DIAGNOSTICO").alias("enfermedad"),
            F.lit("Obesidad").alias("grupo_enfermedad")
        )
    )
    .dropDuplicates()
)

df_plata_enfermedad.show(10)

[Stage 37:=======================================>                (17 + 7) / 24]

+--------------+--------------------+----------------+
|cod_enfermedad|          enfermedad|grupo_enfermedad|
+--------------+--------------------+----------------+
|         E13.8|DIABETES MELLITUS...|        Diabetes|
|         E14.6|DIABETES MELLITUS...|        Diabetes|
|         E10.9|DIABETES MELLITUS...|        Diabetes|
|         E12.8|DIABETES MELLITUS...|        Diabetes|
|         E11.1|DIABETES MELLITUS...|        Diabetes|
|         E12.5|DIABETES MELLITUS...|        Diabetes|
|         E16.2|HIPOGLICEMIA, NO ...|        Diabetes|
|         E10.4|DIABETES MELLITUS...|        Diabetes|
|         E16.8|OTROS TRASTORNOS ...|        Diabetes|
|         E11.9|DIABETES MELLITUS...|        Diabetes|
+--------------+--------------------+----------------+
only showing top 10 rows



### Ubigeo

In [21]:
df_plata_ubigeo = (
    df_bronce_diabetes.select(
        F.col("UBIGEO").alias("cod_ubigeo"),
        F.col("DEPARTAMENTO").alias("departamento"),
        F.col("PROVINCIA").alias("provincia"),
        F.col("DISTRITO").alias("distrito")
    )
    .union(
        df_bronce_hipertension.select(
            F.col("UBIGEO").alias("cod_ubigeo"),
            F.col("DEPARTAMENTO").alias("departamento"),
            F.col("PROVINCIA").alias("provincia"),
            F.col("DISTRITO").alias("distrito")
        )
    )
    .union(
        df_bronce_obesidad.select(
            F.col("UBIGEO").alias("cod_ubigeo"),
            F.col("DEPARTAMENTO").alias("departamento"),
            F.col("PROVINCIA").alias("provincia"),
            F.col("DISTRITO").alias("distrito")
        )
    )
    .dropDuplicates()
)

df_plata_ubigeo.show(10)

+----------+------------+---------+----------------+
|cod_ubigeo|departamento|provincia|        distrito|
+----------+------------+---------+----------------+
|    150101|        LIMA|     LIMA|            LIMA|
|    230401|       TACNA|   TARATA|          TARATA|
|    210201|        PUNO| AZANGARO|        AZANGARO|
|     40110|    AREQUIPA| AREQUIPA|      MIRAFLORES|
|     21806|      ANCASH|    SANTA|          NEPENA|
|    150605|        LIMA|   HUARAL|         CHANCAY|
|     20601|      ANCASH|  CARHUAZ|         CARHUAZ|
|    110508|         ICA|    PISCO|TUPAC AMARU INCA|
|    130202| LA LIBERTAD|   ASCOPE|         CHICAMA|
|    130102| LA LIBERTAD| TRUJILLO|     EL PORVENIR|
+----------+------------+---------+----------------+
only showing top 10 rows



### Diagnostico y Resultado

In [23]:
df_diagnostico_union = (
    df_bronce_diabetes.select(
        F.col("COD_DIAG").alias("cod_enfermedad"),
        F.col("ID_PACIENTE").alias("cod_paciente"),
        F.col("ID_MEDICO").alias("cod_medico"),
        F.col("UBIGEO").alias("cod_ubigeo"),
        F.col("SERVICIO_HOSPITALARIO").alias("servicio_hospitalario"),
        F.col("ACTIVIDAD_HOSPITALARIA").alias("actividad_hospitalaria"),
        F.col("FECHA_MUESTRA").alias("fecha_muestra")
    )
    .unionByName(
        df_bronce_hipertension.select(
            F.col("COD_DIAG").alias("cod_enfermedad"),
            F.col("ID_PACIENTE").alias("cod_paciente"),
            F.col("ID_MEDICO").alias("cod_medico"),
            F.col("UBIGEO").alias("cod_ubigeo"),
            F.col("SERVICIO_HOSPITALARIO").alias("servicio_hospitalario"),
            F.col("ACTIVIDAD_HOSPITALARIA").alias("actividad_hospitalaria"),
            F.col("FECHA_MUESTRA").alias("fecha_muestra")
        )
    )
    .unionByName(
        df_bronce_obesidad.select(
            F.col("COD_DIAG").alias("cod_enfermedad"),
            F.col("ID_PACIENTE").alias("cod_paciente"),
            F.col("ID_MEDICO").alias("cod_medico"),
            F.col("UBIGEO").alias("cod_ubigeo"),
            F.col("SERVICIO_HOSPITALARIO").alias("servicio_hospitalario"),
            F.col("ACTIVIDAD_HOSPITALARIA").alias("actividad_hospitalaria"),
            F.col("FECHA_MUESTRA").alias("fecha_muestra")
        )
    )
)

# Window para ID incremental
w = Window.orderBy(F.monotonically_increasing_id())

df_plata_diagnostico = df_diagnostico_union.withColumn(
    "cod_diagnostico",
    F.row_number().over(w)
)

df_plata_diagnostico = df_plata_diagnostico.select(
    "cod_diagnostico",
    "cod_enfermedad",
    "cod_paciente",
    "cod_medico",
    "cod_ubigeo",
    "servicio_hospitalario",
    "actividad_hospitalaria",
    "fecha_muestra"
)

df_plata_diagnostico.show(10)
df_plata_diagnostico.printSchema()

25/12/10 04:42:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 04:42:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 04:42:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
[Stage 43:=================================================>      (21 + 3) / 24]

+---------------+--------------+--------------------+--------------------+----------+---------------------+----------------------+-------------+
|cod_diagnostico|cod_enfermedad|        cod_paciente|          cod_medico|cod_ubigeo|servicio_hospitalario|actividad_hospitalaria|fecha_muestra|
+---------------+--------------+--------------------+--------------------+----------+---------------------+----------------------+-------------+
|              1|         E11.9|eJwzNDAwtDC0NDMxN...|eJwzNjA2MzE2NLY0N...|    250107|     MEDICINA GENERAL|  ATENCION  MEDICA ...|     20200102|
|              2|         E13.9|eJwzNDAwMjAxNTUxs...|eJwzsjS1NDI2MjE2N...|    250105|     MEDICINA GENERAL|  ATENCION  MEDICA ...|     20200102|
|              3|         E11.9|eJwzNDCwMLOwMDKzt...|eJwztLQwNjM1NTY3N...|    230103|       ENDOCRINOLOGIA|  ATENCION  MEDICA ...|     20200102|
|              4|         E11.9|eJwzNDCwNDU3Mja1M...|eJwztLS0MDE1sjQ3N...|    230101| MEDICINA FAMILIAR...|  ATENCION  MEDICA ...|

In [26]:
# DIABETES
df_join_diabetes = (
    df_bronce_diabetes.join(
        df_plata_diagnostico,
        (
            (df_bronce_diabetes.COD_DIAG == df_plata_diagnostico.cod_enfermedad) &
            (df_bronce_diabetes.ID_PACIENTE == df_plata_diagnostico.cod_paciente) &
            (df_bronce_diabetes.ID_MEDICO == df_plata_diagnostico.cod_medico) &
            (df_bronce_diabetes.UBIGEO == df_plata_diagnostico.cod_ubigeo) &
            (df_bronce_diabetes.FECHA_MUESTRA == df_plata_diagnostico.fecha_muestra)
        ),
        "inner"
    )
    .select(
        "cod_diagnostico",
        "PROCEDIMIENTO_1", "RESULTADO_1", "UNIDADES_1", "FEC_RESULTADO_1",
        "PROCEDIMIENTO_2", "RESULTADO_2", "UNIDADES_2", "FEC_RESULTADO_2"
    )
)

# HIPERTENSION
df_join_hipertension = (
    df_bronce_hipertension.join(
        df_plata_diagnostico,
        (
            (df_bronce_hipertension.COD_DIAG == df_plata_diagnostico.cod_enfermedad) &
            (df_bronce_hipertension.ID_PACIENTE == df_plata_diagnostico.cod_paciente) &
            (df_bronce_hipertension.ID_MEDICO == df_plata_diagnostico.cod_medico) &
            (df_bronce_hipertension.UBIGEO == df_plata_diagnostico.cod_ubigeo) &
            (df_bronce_hipertension.FECHA_MUESTRA == df_plata_diagnostico.fecha_muestra)
        ),
        "inner"
    )
    .select(
        "cod_diagnostico",
        "PROCEDIMIENTO_1", "RESULTADO_1", "UNIDADES_1", "FEC_RESULTADO_1",
        "PROCEDIMIENTO_2", "RESULTADO_2", "UNIDADES_2", "FEC_RESULTADO_2"
    )
)

# OBESIDAD
df_join_obesidad = (
    df_bronce_obesidad.join(
        df_plata_diagnostico,
        (
            (df_bronce_obesidad.COD_DIAG == df_plata_diagnostico.cod_enfermedad) &
            (df_bronce_obesidad.ID_PACIENTE == df_plata_diagnostico.cod_paciente) &
            (df_bronce_obesidad.ID_MEDICO == df_plata_diagnostico.cod_medico) &
            (df_bronce_obesidad.UBIGEO == df_plata_diagnostico.cod_ubigeo) &
            (df_bronce_obesidad.FECHA_MUESTRA == df_plata_diagnostico.fecha_muestra)
        ),
        "inner"
    )
    .select(
        "cod_diagnostico",
        "PROCEDIMIENTO_1", "RESULTADO_1", "UNIDADES_1", "FEC_RESULTADO_1",
        "PROCEDIMIENTO_2", "RESULTADO_2", "UNIDADES_2", "FEC_RESULTADO_2"
    )
)


df_join_all = df_join_diabetes.unionByName(df_join_hipertension).unionByName(df_join_obesidad)

df_proc_1 = df_join_all.select(
    F.col("cod_diagnostico"),
    F.col("PROCEDIMIENTO_1").alias("des_procedimiento"),
    F.col("RESULTADO_1").alias("resultado"),
    F.col("UNIDADES_1").alias("unidades"),
    F.col("FEC_RESULTADO_1").alias("fecha_resultado")
).filter(F.col("des_procedimiento").isNotNull())

df_proc_2 = df_join_all.select(
    F.col("cod_diagnostico"),
    F.col("PROCEDIMIENTO_2").alias("des_procedimiento"),
    F.col("RESULTADO_2").alias("resultado"),
    F.col("UNIDADES_2").alias("unidades"),
    F.col("FEC_RESULTADO_2").alias("fecha_resultado")
).filter(F.col("des_procedimiento").isNotNull())

df_procedimientos = df_proc_1.unionByName(df_proc_2)

df_plata_resultado_procedimiento = df_procedimientos.join(
    df_plata_procedimiento,
    df_procedimientos.des_procedimiento == df_plata_procedimiento.des_procedimiento,
    "left"
).select(
    "cod_procedimiento",
    "cod_diagnostico",
    "resultado",
    "unidades",
    "fecha_resultado"
)

df_plata_resultado_procedimiento.show(10)

25/12/10 04:50:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 04:50:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 04:50:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 04:50:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 04:50:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 04:50:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 0

+-----------------+---------------+---------+--------+---------------+
|cod_procedimiento|cod_diagnostico|resultado|unidades|fecha_resultado|
+-----------------+---------------+---------+--------+---------------+
|                1|              1|    217.0|   mg/dL|       20200111|
|                1|              2|    192.0|   mg/dL|       20200111|
|                1|              3|    203.0|   mg/dL|       20200102|
|                1|              4|    207.0|   mg/dL|       20200102|
|                1|              5|    155.0|   mg/dL|       20200102|
|                3|              6|    149.0|   mg/dL|       20200102|
|                1|              7|    221.0|   mg/dL|       20200102|
|                1|              8|    152.0|   mg/dL|       20200102|
|                3|              9|    221.0|   mg/dL|       20200106|
|                1|             10|    126.0|   mg/dL|       20200102|
+-----------------+---------------+---------+--------+---------------+
only s

In [32]:
df_plata_resultado_procedimiento.orderBy("cod_diagnostico").show(10)
df_plata_resultado_procedimiento.printSchema()

25/12/10 04:56:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 04:56:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 04:56:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 04:56:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 04:56:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 04:56:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 0

+-----------------+---------------+---------+--------+---------------+
|cod_procedimiento|cod_diagnostico|resultado|unidades|fecha_resultado|
+-----------------+---------------+---------+--------+---------------+
|                3|              1|    258.0|   mg/dL|       20200111|
|                1|              1|    217.0|   mg/dL|       20200111|
|                3|              2|    171.0|   mg/dL|       20200111|
|                1|              2|    192.0|   mg/dL|       20200111|
|                3|              3|     78.0|   mg/dL|       20200102|
|                1|              3|    203.0|   mg/dL|       20200102|
|                3|              4|    113.0|   mg/dL|       20200102|
|                1|              4|    207.0|   mg/dL|       20200102|
|                3|              5|     78.0|   mg/dL|       20200102|
|                1|              5|    155.0|   mg/dL|       20200102|
+-----------------+---------------+---------+--------+---------------+
only s

### Ajuste de Fechas

In [33]:
from pyspark.sql.functions import to_date, col

df_plata_resultado_procedimiento = df_plata_resultado_procedimiento.withColumn(
    "fecha_resultado",
    to_date(col("fecha_resultado").cast("string"), "yyyyMMdd")
)

df_plata_diagnostico = df_plata_diagnostico.withColumn(
    "fecha_muestra",
    to_date(col("fecha_muestra").cast("string"), "yyyyMMdd")
)


## Ahora subiendo a BigQuery los datos Plata

In [36]:
df_plata_procedimiento.write.format("bigquery") \
    .option("table", "grupo2-essalud.plata.procedimiento") \
    .option("temporaryGcsBucket", "grupo2-essalud-datalake") \
    .mode("overwrite") \
    .save()

df_plata_medico.write.format("bigquery") \
    .option("table", "grupo2-essalud.plata.medico") \
    .option("temporaryGcsBucket", "grupo2-essalud-datalake") \
    .mode("overwrite") \
    .save()

df_plata_paciente.write.format("bigquery") \
    .option("table", "grupo2-essalud.plata.paciente") \
    .option("temporaryGcsBucket", "grupo2-essalud-datalake") \
    .mode("overwrite") \
    .save()

df_plata_enfermedad.write.format("bigquery") \
    .option("table", "grupo2-essalud.plata.enfermedad") \
    .option("temporaryGcsBucket", "grupo2-essalud-datalake") \
    .mode("overwrite") \
    .save()

df_plata_ubigeo.write.format("bigquery") \
    .option("table", "grupo2-essalud.plata.ubigeo") \
    .option("temporaryGcsBucket", "grupo2-essalud-datalake") \
    .mode("overwrite") \
    .save()

df_plata_diagnostico.write.format("bigquery") \
    .option("table", "grupo2-essalud.plata.diagnostico") \
    .option("temporaryGcsBucket", "grupo2-essalud-datalake") \
    .mode("overwrite") \
    .save()

df_plata_resultado_procedimiento.write.format("bigquery") \
    .option("table", "grupo2-essalud.plata.resultado_procedimiento") \
    .option("temporaryGcsBucket", "grupo2-essalud-datalake") \
    .mode("overwrite") \
    .save()

25/12/10 05:07:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 05:07:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 05:07:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 05:07:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 05:07:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 05:07:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 0

## Guardando los Plata como csv

In [37]:
!pip install google-cloud-storage

Looking in indexes: https://us-python.pkg.dev/artifact-registry-python-cache/virtual-python/simple/
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.17.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 6.33.2 which is incompatible.
spacy 3.8.7 requires thinc<8.4.0,>=8.3.4, but you have thinc 8.3.2 which is incompatible.
pyiceberg 0.9.0 requires cachetools<6.0.0,>=5.5.0, but you have cachetools 6.2.2 which is incompatible.
grpcio-status 1.65.5 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.2 which is incompatible.
ydata-profiling 0.0.dev0 requires numba<=0.61,>=0.56.0, but you have numba 0.61.2 which is incompatible.
google-auth-oauthlib 1.2.3 requires google-auth<2.42.0,>=2.15.0, but you have google-auth 2.43.0 which is incompatible.
google-cloud-aiplatform 1.94.0 requir

In [40]:
from google.cloud import storage

def rename_gcs_file(bucket_name, source_path, destination_path):
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    source_blob = bucket.blob(source_path)
    bucket.copy_blob(source_blob, bucket, destination_path)
    source_blob.delete()
    print(f"Renombrado: {source_path}  →  {destination_path}")

def export_single_csv(df, bucket, folder, final_filename):
    output_path = f"gs://{bucket}/{folder}/temp_export"
    
    df.coalesce(1).write \
        .option("header", "true") \
        .mode("overwrite") \
        .csv(output_path)

    client = storage.Client()
    bucket_ref = client.bucket(bucket)
    blobs = list(bucket_ref.list_blobs(prefix=f"{folder}/temp_export/"))

    part_file = None
    for b in blobs:
        if b.name.endswith(".csv"):
            part_file = b.name
            break

    if part_file is None:
        raise Exception("No se encontró el archivo CSV exportado.")

    final_path = f"{folder}/{final_filename}"
    rename_gcs_file(bucket, part_file, final_path)

    for b in blobs:
        try:
            b.delete()
        except Exception:
            pass  # si ya fue borrado, seguimos normal

    print(f"Archivo final generado: gs://{bucket}/{final_path}")


In [41]:
bucket = "grupo2-essalud-datalake"

export_single_csv(df_plata_procedimiento, bucket, "plata/procedimiento", "procedimiento.csv")
export_single_csv(df_plata_medico, bucket, "plata/medico", "medico.csv")
export_single_csv(df_plata_paciente, bucket, "plata/paciente", "paciente.csv")
export_single_csv(df_plata_enfermedad, bucket, "plata/enfermedad", "enfermedad.csv")
export_single_csv(df_plata_ubigeo, bucket, "plata/ubigeo", "ubigeo.csv")
export_single_csv(df_plata_diagnostico, bucket, "plata/diagnostico", "diagnostico.csv")
export_single_csv(df_plata_resultado_procedimiento, bucket, "plata/resultado_procedimiento", "resultado_procedimiento.csv")

25/12/10 05:18:25 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 05:18:25 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 05:18:25 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 05:18:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 05:18:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 05:18:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 0

Renombrado: plata/procedimiento/temp_export/part-00000-52005628-91aa-4c37-a9bd-9b2dc4d59b00-c000.csv  →  plata/procedimiento/procedimiento.csv
Archivo final generado: gs://grupo2-essalud-datalake/plata/procedimiento/procedimiento.csv


Renombrado: plata/medico/temp_export/part-00000-a3b5fa3e-b044-444b-9880-009b8495a018-c000.csv  →  plata/medico/medico.csv
Archivo final generado: gs://grupo2-essalud-datalake/plata/medico/medico.csv


Renombrado: plata/paciente/temp_export/part-00000-0bb7b011-bd2a-491e-a712-a91ff70aeee9-c000.csv  →  plata/paciente/paciente.csv
Archivo final generado: gs://grupo2-essalud-datalake/plata/paciente/paciente.csv


Renombrado: plata/enfermedad/temp_export/part-00000-254eef42-8427-4ea5-ad67-e0af3134fca0-c000.csv  →  plata/enfermedad/enfermedad.csv
Archivo final generado: gs://grupo2-essalud-datalake/plata/enfermedad/enfermedad.csv


Renombrado: plata/ubigeo/temp_export/part-00000-7f3bfaf5-e6e9-4c4c-9742-3c7aa0b35f5c-c000.csv  →  plata/ubigeo/ubigeo.csv
Archivo final generado: gs://grupo2-essalud-datalake/plata/ubigeo/ubigeo.csv


25/12/10 05:18:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 05:18:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 05:18:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 05:18:51 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 05:18:51 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
                                                                                

Renombrado: plata/diagnostico/temp_export/part-00000-f5768aa7-11d1-4c8b-928b-5efdd223e9cb-c000.csv  →  plata/diagnostico/diagnostico.csv
Archivo final generado: gs://grupo2-essalud-datalake/plata/diagnostico/diagnostico.csv


25/12/10 05:19:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 05:19:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 05:19:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 05:19:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 05:19:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 05:19:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 0

Renombrado: plata/resultado_procedimiento/temp_export/part-00000-ffb9cd3d-68f2-4b2d-8b83-0c4e7f78b49a-c000.csv  →  plata/resultado_procedimiento/resultado_procedimiento.csv
Archivo final generado: gs://grupo2-essalud-datalake/plata/resultado_procedimiento/resultado_procedimiento.csv


# De Plata a Oro

In [43]:
df_plata_procedimiento.printSchema()
df_plata_medico.printSchema()
df_plata_paciente.printSchema()
df_plata_enfermedad.printSchema()
df_plata_ubigeo.printSchema()
df_plata_diagnostico.printSchema()
df_plata_resultado_procedimiento.printSchema()

root
 |-- des_procedimiento: string (nullable = true)
 |-- cod_procedimiento: integer (nullable = false)

root
 |-- cod_medico: string (nullable = true)
 |-- edad_medico: integer (nullable = true)

root
 |-- cod_paciente: string (nullable = true)
 |-- edad_paciente: integer (nullable = true)
 |-- sexo_paciente: string (nullable = true)

root
 |-- cod_enfermedad: string (nullable = true)
 |-- enfermedad: string (nullable = true)
 |-- grupo_enfermedad: string (nullable = false)

root
 |-- cod_ubigeo: integer (nullable = true)
 |-- departamento: string (nullable = true)
 |-- provincia: string (nullable = true)
 |-- distrito: string (nullable = true)

root
 |-- cod_diagnostico: integer (nullable = false)
 |-- cod_enfermedad: string (nullable = true)
 |-- cod_paciente: string (nullable = true)
 |-- cod_medico: string (nullable = true)
 |-- cod_ubigeo: integer (nullable = true)
 |-- servicio_hospitalario: string (nullable = true)
 |-- actividad_hospitalaria: string (nullable = true)
 |-- fec

### Dim_Tiempo

In [45]:
from pyspark.sql import functions as F

df_fechas = (
    df_plata_diagnostico.select(F.col("fecha_muestra").alias("fecha"))
    .union(df_plata_resultado_procedimiento.select(F.col("fecha_resultado").alias("fecha")))
    .dropna()
    .distinct()
)

df_oro_tiempo = (
    df_fechas
    .withColumn("SK_Tiempo", F.monotonically_increasing_id())
    .withColumn("año", F.year("fecha"))
    .withColumn("mes", F.month("fecha"))
    .withColumn("dia", F.dayofmonth("fecha"))
    .withColumn("semana", F.weekofyear("fecha"))
    .withColumn("trimestre", F.quarter("fecha"))
    .withColumn("fin_de_mes", F.last_day("fecha"))
)


### Dim_Paciente

In [46]:
df_oro_paciente = (
    df_plata_paciente
    .withColumn("SK_Paciente", F.monotonically_increasing_id())
    .withColumn(
        "grupo_etario",
        F.when(F.col("edad_paciente") < 18, "Menor")
         .when(F.col("edad_paciente") < 60, "Adulto")
         .otherwise("Adulto Mayor")
    )
)

### Dim_Enfermedad

In [49]:
df_oro_enfermedad = (
    df_plata_enfermedad
    .withColumn("SK_Enfermedad", F.monotonically_increasing_id())
    .withColumnRenamed("enfermedad", "des_enfermedad")
)

### Dim_Ubigeo

In [50]:
df_oro_ubigeo = (
    df_plata_ubigeo
    .withColumn("SK_Ubigeo", F.monotonically_increasing_id())
    .withColumnRenamed("cod_ubigeo", "ubigeo")
    .withColumn("macroRegion",
                F.when(F.col("departamento").isin("LIMA", "CALLAO"), "Costa Central")
                 .otherwise("Otra"))
)

### Dim_Procedimiento

In [55]:
df_resul_enriched = (
    df_plata_resultado_procedimiento
        .join(
            df_plata_procedimiento,
            "cod_procedimiento",
            "left"
        )
)

df_oro_procedimiento = (
    df_resul_enriched
        .select("des_procedimiento", "unidades")
        .distinct()
        .withColumn("SK_Procedimiento", F.monotonically_increasing_id())
)

### Fact_Diagnostico

In [59]:
# Join con dimensiones para traer SKs
df_oro_fact_diagnostico = (
    df_plata_diagnostico
    # Tiempo
    .join(df_oro_tiempo.select("fecha", "SK_Tiempo"),
          df_plata_diagnostico.fecha_muestra == df_oro_tiempo.fecha,
          "left")
    # Paciente
    .join(df_oro_paciente.select("cod_paciente", "SK_Paciente"),
          "cod_paciente",
          "left")
    # Enfermedad
    .join(df_oro_enfermedad.select("cod_enfermedad", "SK_Enfermedad"),
          "cod_enfermedad",
          "left")
    # Ubigeo
    .join(df_oro_ubigeo.select("ubigeo", "SK_Ubigeo"),
          df_plata_diagnostico.cod_ubigeo == df_oro_ubigeo.ubigeo,
          "left")
    .withColumn("SK_Diagnostico", F.monotonically_increasing_id())
    .select(
        F.col("cod_diagnostico"),
        "SK_Diagnostico",
        "SK_Tiempo",
        "SK_Paciente",
        "SK_Enfermedad",
        "SK_Ubigeo",
        "servicio_hospitalario",
        "actividad_hospitalaria"
    )
)

### Fact_Resultado

In [61]:
df_oro_fact_resultado = (
    df_resul_enriched
        .join(
            df_oro_procedimiento,
            ["des_procedimiento", "unidades"],
            "left"
        )
        .join(
            df_oro_tiempo.select("fecha", "SK_Tiempo"),
            df_resul_enriched.fecha_resultado == F.col("fecha"),
            "left"
        )
        .join(
            df_oro_fact_diagnostico.select("SK_Diagnostico", "cod_diagnostico"),
            "cod_diagnostico",
            "left"
        )
        .withColumn("SK_Resultado", F.monotonically_increasing_id())
        .select(
            "SK_Resultado",
            "SK_Tiempo",
            "SK_Diagnostico",
            F.col("resultado").alias("medida_resultado")
        )
)


## Carga a BigQuery

In [62]:
df_oro_tiempo.write.format("bigquery") \
    .option("table", "grupo2-essalud.oro.dim_tiempo") \
    .option("temporaryGcsBucket", "grupo2-essalud-datalake") \
    .mode("overwrite") \
    .save()

df_oro_paciente.write.format("bigquery") \
    .option("table", "grupo2-essalud.oro.dim_paciente") \
    .option("temporaryGcsBucket", "grupo2-essalud-datalake") \
    .mode("overwrite") \
    .save()

df_oro_ubigeo.write.format("bigquery") \
    .option("table", "grupo2-essalud.oro.dim_ubigeo") \
    .option("temporaryGcsBucket", "grupo2-essalud-datalake") \
    .mode("overwrite") \
    .save()

df_oro_procedimiento.write.format("bigquery") \
    .option("table", "grupo2-essalud.oro.dim_procedimiento") \
    .option("temporaryGcsBucket", "grupo2-essalud-datalake") \
    .mode("overwrite") \
    .save()

df_oro_fact_diagnostico.write.format("bigquery") \
    .option("table", "grupo2-essalud.oro.fact_diagnostico") \
    .option("temporaryGcsBucket", "grupo2-essalud-datalake") \
    .mode("overwrite") \
    .save()

df_oro_fact_resultado.write.format("bigquery") \
    .option("table", "grupo2-essalud.oro.fact_resultado") \
    .option("temporaryGcsBucket", "grupo2-essalud-datalake") \
    .mode("overwrite") \
    .save()

25/12/10 05:55:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 05:55:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 05:55:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 05:55:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 05:55:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 05:55:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 0

In [64]:
df_oro_enfermedad.write.format("bigquery") \
    .option("table", "grupo2-essalud.oro.dim_enfermedad") \
    .option("temporaryGcsBucket", "grupo2-essalud-datalake") \
    .mode("overwrite") \
    .save()

## Guardando los Oro como csv

In [63]:
bucket = "grupo2-essalud-datalake"

export_single_csv(df_oro_tiempo, bucket, "oro/dim_tiempo", "dim_tiempo.csv")
export_single_csv(df_oro_paciente, bucket, "oro/dim_paciente", "dim_paciente.csv")
export_single_csv(df_oro_ubigeo, bucket, "oro/dim_ubigeo", "dim_ubigeo.csv")
export_single_csv(df_oro_procedimiento, bucket, "oro/dim_procedimiento", "dim_procedimiento.csv")
export_single_csv(df_oro_fact_diagnostico, bucket, "oro/fact_diagnostico", "fact_diagnostico.csv")
export_single_csv(df_oro_fact_resultado, bucket, "oro/fact_resultado", "fact_resultado.csv")

Renombrado: oro/dim_tiempo/temp_export/part-00000-6cc562e0-d5d1-4dd7-8881-cc14ceedb32b-c000.csv  →  oro/dim_tiempo/dim_tiempo.csv
Archivo final generado: gs://grupo2-essalud-datalake/oro/dim_tiempo/dim_tiempo.csv


Renombrado: oro/dim_paciente/temp_export/part-00000-ed8c240f-a368-47fa-b6a4-460b5681a7bc-c000.csv  →  oro/dim_paciente/dim_paciente.csv
Archivo final generado: gs://grupo2-essalud-datalake/oro/dim_paciente/dim_paciente.csv


Renombrado: oro/dim_ubigeo/temp_export/part-00000-c00c7291-ed9a-4225-8f27-29ce75f7597d-c000.csv  →  oro/dim_ubigeo/dim_ubigeo.csv
Archivo final generado: gs://grupo2-essalud-datalake/oro/dim_ubigeo/dim_ubigeo.csv


25/12/10 06:03:48 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 06:03:48 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 06:03:48 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 06:03:48 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 06:03:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 06:03:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 0

Renombrado: oro/dim_procedimiento/temp_export/part-00000-a88dab57-292d-4181-8daa-27bd20eb42e6-c000.csv  →  oro/dim_procedimiento/dim_procedimiento.csv
Archivo final generado: gs://grupo2-essalud-datalake/oro/dim_procedimiento/dim_procedimiento.csv


25/12/10 06:04:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 06:04:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 06:04:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 06:04:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 06:04:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 06:04:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 0

Renombrado: oro/fact_diagnostico/temp_export/part-00000-dd0be8bd-36ec-486e-ae6a-110b5ef2027a-c000.csv  →  oro/fact_diagnostico/fact_diagnostico.csv
Archivo final generado: gs://grupo2-essalud-datalake/oro/fact_diagnostico/fact_diagnostico.csv


25/12/10 06:05:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 06:05:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 06:05:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 06:05:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 06:05:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 06:05:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/10 0

Renombrado: oro/fact_resultado/temp_export/part-00000-9535616e-5680-4000-bcf7-d6b39f8e8643-c000.csv  →  oro/fact_resultado/fact_resultado.csv
Archivo final generado: gs://grupo2-essalud-datalake/oro/fact_resultado/fact_resultado.csv


In [65]:
export_single_csv(df_oro_enfermedad, bucket, "oro/dim_enfermedad", "dim_enfermedad.csv")

Renombrado: oro/dim_enfermedad/temp_export/part-00000-6c0192b8-75ec-48c3-9a6a-b22a891b74d4-c000.csv  →  oro/dim_enfermedad/dim_enfermedad.csv
Archivo final generado: gs://grupo2-essalud-datalake/oro/dim_enfermedad/dim_enfermedad.csv
